# W23 · TF2 坐标变换与 URDF 机器人描述

> 机器人系统里每个传感器、每个关节都活在自己的坐标系里。
> TF2 负责回答「这个点在世界系下的坐标是什么」；URDF 负责描述「机器人长什么样」。
> 这一讲是阶段三数学含量最高的一讲——好消息是，你在阶段一打的线性代数基础够用了。

## 学习目标

1. 掌握旋转的三种表示（旋转矩阵 / 欧拉角 / 四元数）及互换，理解万向锁；
2. 会用齐次变换矩阵做坐标系链式复合 $T_{AC} = T_{AB}\,T_{BC}$；
3. 理解 TF2 的「坐标树」模型，会广播/监听变换；
4. 手写一个差速小车的 URDF，并用 Xacro 消除重复；
5. 能用 `robot_state_publisher` + `joint_state_publisher` 把 URDF 变成 TF 树。

## ⚠️ 运行前提

- `transforms3d` 与 URDF 解析演示为纯 Python，本机真实执行（`uv sync --extra ros` 后可用）；
- `ros2 run tf2_ros ...`、`robot_state_publisher` 等命令需在 ROS2 Jazzy 环境执行（标注前提的 cell 不执行）。

## 1. 旋转表示：从欧拉角的坑到四元数

**直觉**：描述「朝向」最自然的方式是欧拉角（roll/pitch/yaw，绕 $x/y/z$ 轴转多少）。
但欧拉角有致命缺陷——**万向锁（Gimbal Lock）**：当中间轴转到 $\pm 90°$ 时，
两个旋转轴重合，损失一个自由度，插值和求导都会出奇点。

**类比**：欧拉角像「先右转 90°、再抬头 45°」的口头指令——顺序不同结果不同；
四元数则像一个四维单位向量 $q = (w, x, y, z)$，直接编码「绕轴 $\hat{n}$ 转 $\theta$」：

$$
q = \left(\cos\tfrac{\theta}{2},\ \hat{n}\sin\tfrac{\theta}{2}\right), \qquad \|q\| = 1
$$

工程事实：ROS2 所有消息里的姿态（`geometry_msgs/Quaternion`）**一律用四元数**。
你不需要手算四元数乘法，但要会转换和检查合法性（模长必须为 1）。

In [1]:
"""用 transforms3d 玩转旋转表示（纯 Python，本机执行）。"""
import numpy as np
from transforms3d.euler import euler2quat, quat2euler
from transforms3d.quaternions import qmult, quat2mat

# --- 1) 欧拉角 <-> 四元数（transforms3d 默认 'sxyz' 静态轴，即 ROS 的固定轴 RPY 约定）---
roll, pitch, yaw = np.deg2rad([30, 45, 60])
q = euler2quat(roll, pitch, yaw, axes="sxyz")
print("RPY(deg) =", [30, 45, 60])
print("四元数 q =", np.round(q, 4), " 模长 =", np.linalg.norm(q).round(6))
r, p, y = quat2euler(q, axes="sxyz")
print("转回 RPY(deg) =", np.round(np.rad2deg([r, p, y]), 4))

# --- 2) 四元数 -> 旋转矩阵，验证 R 是正交矩阵：R^T R = I ---
R = quat2mat(q)
print("\nR^T R ≈ I ?", np.allclose(R.T @ R, np.eye(3)), " det(R) =", np.linalg.det(R).round(6))

# --- 3) 四元数乘法 = 旋转复合（注意：不可交换！）---
q_yaw90 = euler2quat(0, 0, np.pi / 2)      # 先绕 z 转 90°
q_roll90 = euler2quat(np.pi / 2, 0, 0)     # 再绕 x 转 90°
v = np.array([1.0, 0.0, 0.0])
v_ab = quat2mat(qmult(q_roll90, q_yaw90)) @ v   # 先 yaw 后 roll
v_ba = quat2mat(qmult(q_yaw90, q_roll90)) @ v   # 先 roll 后 yaw
print("\n同一向量两种顺序旋转结果：")
print("  先yaw后roll:", np.round(v_ab, 4), " 先roll后yaw:", np.round(v_ba, 4), " 相等?", np.allclose(v_ab, v_ba))

RPY(deg) = [30, 45, 60]
四元数 q = [0.8224 0.0223 0.4397 0.3604]  模长 = 1.0
转回 RPY(deg) = [30. 45. 60.]

R^T R ≈ I ? True  det(R) = 1.0

同一向量两种顺序旋转结果：
  先yaw后roll: [0. 0. 1.]  先roll后yaw: [0. 1. 0.]  相等? False


**观察**：四元数模长恒为 1；旋转矩阵满足 $R^\top R = I$、$\det R = +1$；
两种复合顺序结果不同——**旋转不可交换**，这就是 TF2 里父-子方向绝不能搞反的原因。

## 2. 齐次变换：旋转 + 平移一把抓

仅旋转不够——传感器相对机器人还有**位置**偏移。把旋转 $R$ 和平移 $t$ 装进 4×4 矩阵：

$$
T = \begin{bmatrix} R & t \\ \mathbf{0} & 1 \end{bmatrix}, \qquad
p_A = T_{AB}\, p_B
$$

链式法则：$T_{AC} = T_{AB}\, T_{BC}$。求逆有闭式：$T^{-1} = \begin{bmatrix} R^\top & -R^\top t \\ 0 & 1\end{bmatrix}$。

**这正是 TF2 的全部数学**：TF2 维护一棵坐标系树，每条边是一个 $T$，
查询「A 到 C 的变换」就是把路径上的矩阵连乘。

In [2]:
"""齐次变换与链式复合：激光雷达测到的障碍物点 → 世界坐标。

场景：base_link 相对 world 位于 (1, 2, yaw=90°)；lidar 装在 base_link 前方 0.3 m；
lidar 测得障碍物在自身坐标系 (2.0, 0.1, 0)。求障碍物的 world 坐标。
"""
import numpy as np
from transforms3d.euler import euler2mat


def make_T(translation, rpy):
    """由平移 + RPY 构造齐次变换矩阵。"""
    R = euler2mat(*rpy, axes="sxyz")
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = translation
    return T

T_world_base = make_T([1.0, 2.0, 0.0], [0, 0, np.pi / 2])
T_base_lidar = make_T([0.3, 0.0, 0.2], [0, 0, 0])

T_world_lidar = T_world_base @ T_base_lidar          # 链式复合
p_lidar = np.array([2.0, 0.1, 0.0, 1.0])             # 齐次坐标
p_world = T_world_lidar @ p_lidar
print("障碍物 world 坐标:", np.round(p_world[:3], 4))

# 验证逆变换：从 world 反推回 lidar 系，应还原 (2.0, 0.1, 0)
print("逆变换还原:", np.round((np.linalg.inv(T_world_lidar) @ p_world)[:3], 4))

# 手算验证：base 在世界系朝 +y 方向（yaw=90°），lidar 在 base 前方 0.3m
# 障碍物在 lidar 前方 2.0m、左 0.1m → world 中应为 x=1-2.1+? ... 用脑内小图核对上面的数值

障碍物 world 坐标: [0.9 4.3 0.2]
逆变换还原: [2.  0.1 0. ]


**自查**：`base_link` 朝向 $+y$（yaw 90°），lidar 在 base 前方 0.3 m、高 0.2 m，
障碍物在 lidar 前方 2.0 m、左偏 0.1 m。base 的「前方」是世界的 $+y$，
lidar 的「左」是世界的 $-x$，所以答案约 $(1 - 0.1,\ 2 + 0.3 + 2.0,\ 0.2)$——
和矩阵算出的结果一致，说明你真正理解了链式变换。

## 3. TF2：运行时的坐标树服务

TF2 把上面的矩阵链做成了**分布式在线服务**：

- 任意节点都可以**广播**一条边（`TransformBroadcaster`），例如轮式里程计节点广播
  `odom → base_link`，IMU 驱动广播 `base_link → imu_link`；
- 任意节点都可以**监听/查询**任意两帧之间的变换（`TransformListener` 内部维护缓存树）；
- 查询自带**时间轴**：`lookup_transform(target, source, time)` 回答「t 时刻的变换」，
  这对把 100 Hz 的点云对齐到 10 Hz 的位姿至关重要。

常用 CLI（在 ROS2 机器上执行）：

```bash
# 打印两帧之间的当前变换（平移 + 四元数）
ros2 run tf2_ros tf2_echo base_link lidar_link

# 生成整棵 TF 树的 PDF 图（frames.pdf）
ros2 run tf2_tools view_frames

# 广播一个静态变换（如传感器安装位置）：x y z yaw pitch roll parent child
ros2 run tf2_ros static_transform_publisher --x 0.3 --y 0 --z 0.2 \
  --yaw 0 --pitch 0 --roll 0 --frame-id base_link --child-frame-id lidar_link
```

> ⚠️ 常见坑：`LookupException` 多半是 (a) 树没连起来（某条边没人广播），
> (b) 时间戳对不上（仿真 `/clock` 与系统时间混用），(c) 拼写大小写。

## 4. URDF：机器人的「身体说明书」

**URDF（Unified Robot Description Format）** 是 XML 格式，描述机器人的：

- `<link>`：刚体——`visual`（外观）、`collision`（碰撞体）、`inertial`（质量/质心/惯性张量）；
- `<joint>`：连杆间的连接——类型有 `fixed / revolute（限位旋转）/ continuous（无限旋转）/ prismatic（滑动）`，
  含 `parent/child/origin/axis/limit`；
- 整棵结构必须是**一棵树**：一个 base link，关节构成无环图。

下面用纯 Python 定义并解析一份差速小车的 URDF（本机执行）——
先验证「解析器视角」，下一节再把它交给 `robot_state_publisher` 变成 TF 树。

In [3]:
"""定义差速小车 URDF，并用 xml.etree 解析、校验树结构（纯 Python，本机执行）。"""
import xml.etree.ElementTree as ET

DIFFBOT_URDF = """<?xml version="1.0"?>
<robot name="diffbot">
  <!-- 底盘 -->
  <link name="base_link">
    <visual><geometry><box size="0.4 0.3 0.1"/></geometry>
      <material name="blue"><color rgba="0 0 0.8 1"/></material></visual>
    <collision><geometry><box size="0.4 0.3 0.1"/></geometry></collision>
    <inertial><mass value="5.0"/>
      <inertia ixx="0.05" ixy="0" ixz="0" iyy="0.1" iyz="0" izz="0.1"/></inertial>
  </link>
  <!-- 左/右驱动轮：continuous 关节可无限旋转 -->
  <link name="left_wheel">
    <visual><geometry><cylinder radius="0.08" length="0.03"/></geometry></visual>
    <inertial><mass value="0.5"/>
      <inertia ixx="0.001" ixy="0" ixz="0" iyy="0.001" iyz="0" izz="0.002"/></inertial>
  </link>
  <link name="right_wheel">
    <visual><geometry><cylinder radius="0.08" length="0.03"/></geometry></visual>
    <inertial><mass value="0.5"/>
      <inertia ixx="0.001" ixy="0" ixz="0" iyy="0.001" iyz="0" izz="0.002"/></inertial>
  </link>
  <!-- 万向轮（从动）：fixed 关节 -->
  <link name="caster">
    <visual><geometry><sphere radius="0.04"/></geometry></visual>
    <inertial><mass value="0.2"/>
      <inertia ixx="0.0001" ixy="0" ixz="0" iyy="0.0001" iyz="0" izz="0.0001"/></inertial>
  </link>

  <joint name="left_wheel_joint" type="continuous">
    <parent link="base_link"/><child link="left_wheel"/>
    <origin xyz="0 0.175 0" rpy="-1.5708 0 0"/>
    <axis xyz="0 0 1"/>
  </joint>
  <joint name="right_wheel_joint" type="continuous">
    <parent link="base_link"/><child link="right_wheel"/>
    <origin xyz="0 -0.175 0" rpy="-1.5708 0 0"/>
    <axis xyz="0 0 1"/>
  </joint>
  <joint name="caster_joint" type="fixed">
    <parent link="base_link"/><child link="caster"/>
    <origin xyz="-0.15 0 -0.02" rpy="0 0 0"/>
  </joint>
</robot>
"""

root = ET.fromstring(DIFFBOT_URDF)
links = {l.get("name") for l in root.findall("link")}
joints = root.findall("joint")
print(f"机器人 {root.get('name')}: {len(links)} 个 link, {len(joints)} 个 joint")

# --- 校验：每个 joint 的 parent/child 必须存在；结构必须是树 ---
child_of = {}
for j in joints:
    parent, child = j.find("parent").get("link"), j.find("child").get("link")
    assert parent in links and child in links, f"joint {j.get('name')} 引用了不存在的 link"
    assert child not in child_of, f"{child} 有多个 parent（不是树！）"
    child_of[child] = parent
    print(f"  {j.get('name'):20s} type={j.get('type'):10s} {parent} -> {child}")

roots = links - set(child_of)
print("根 link（base）:", roots, " 是树?", len(roots) == 1)

# --- 提取静态变换：joint origin 就是 TF 树中那条边的 T ---
j = joints[0]
xyz = [float(v) for v in j.find("origin").get("xyz").split()]
rpy = [float(v) for v in j.find("origin").get("rpy").split()]
print(f"\n{j.get('name')} 的静态边: 平移={xyz}, RPY={rpy}")

机器人 diffbot: 4 个 link, 3 个 joint
  left_wheel_joint     type=continuous base_link -> left_wheel
  right_wheel_joint    type=continuous base_link -> right_wheel
  caster_joint         type=fixed      base_link -> caster
根 link（base）: {'base_link'}  是树? True

left_wheel_joint 的静态边: 平移=[0.0, 0.175, 0.0], RPY=[-1.5708, 0.0, 0.0]


**观察**：`continuous` 轮子的 `<origin rpy="-1.5708 0 0">` 把圆柱的默认轴线（z 轴）
转到水平方向；`<axis xyz="0 0 1">` 是**关节坐标系内**的转轴。
`robot_state_publisher` 做的事 = 我们上面做的事 + 对非 fixed 关节代入当前角度。

## 5. 从 URDF 到 TF 树：robot_state_publisher

在 ROS2 机器上，三个工具配合让 URDF「活」起来（需安装
`ros-jazzy-robot-state-publisher ros-jazzy-joint-state-publisher-gui ros-jazzy-xacro`）：

```bash
# 1) 校验 URDF 语法
check_urdf robot.urdf          # 来自 liburdfdom-tools（sudo apt install liburdfdom-tools）

# 2) 发布 TF：robot_state_publisher 读 URDF + /joint_states，广播整棵树
ros2 run robot_state_publisher robot_state_publisher --ros-args -p robot_description:="$(cat robot.urdf)"

# 3) 用滑条手动驱动非 fixed 关节（发布 /joint_states）
ros2 run joint_state_publisher_gui joint_state_publisher_gui

# 4) RViz 可视化：Add -> RobotModel / TF
ros2 run rviz2 rviz2
```

**预期**：RViz 中看到蓝色底盘 + 两个轮子；拖动 joint_state_publisher_gui 的滑条，
轮子的 TF 帧随之旋转，`ros2 run tf2_tools view_frames` 生成的 `frames.pdf` 呈现
`base_link → {left_wheel, right_wheel, caster}` 的树。

## 6. Xacro：给 URDF 加「编程能力」

手写 URDF 的痛点：左右轮只有镜像差，却要复制粘贴；改个轮距要改一堆数字。
**Xacro（XML Macros）** 提供变量、数学表达式和宏，是事实标准（`.urdf.xacro` 文件）：

```xml
<?xml version="1.0"?>
<robot xmlns:xacro="http://www.ros.org/wiki/xacro" name="diffbot">
  <xacro:property name="wheel_radius" value="0.08"/>
  <xacro:property name="track" value="0.35"/>          <!-- 轮距：一处改，处处生效 -->

  <xacro:macro name="wheel" params="side sign">
    <link name="${side}_wheel">
      <visual><geometry><cylinder radius="${wheel_radius}" length="0.03"/></geometry></visual>
    </link>
    <joint name="${side}_wheel_joint" type="continuous">
      <parent link="base_link"/><child link="${side}_wheel"/>
      <origin xyz="0 ${sign * track / 2} 0" rpy="-1.5708 0 0"/>
      <axis xyz="0 0 1"/>
    </joint>
  </xacro:macro>

  <xacro:wheel side="left"  sign="1"/>
  <xacro:wheel side="right" sign="-1"/>
</robot>
```

展开为纯 URDF：`xacro robot.urdf.xacro > robot.urdf`。
W24/W25 会在 xacro 里继续挂 Gazebo 插件和 `<ros2_control>` 标签——xacro 的参数化到那时会救命。

## ✏️ 练习

### 练习 1（★，约 15 分钟，纯 Python）：四元数体检

写一个函数 `check_quat(q)`，检查四元数是否单位化（容差 1e-3），不是则归一化并返回修正值。
用它处理 `[(0.5,0.5,0.5,0.5), (1,0,0,1), (0,0,0,0)]`，说明第三个输入为什么是**非法姿态**、
ROS2 消息里收到它会怎样。交付：函数 + 测试输出。

### 练习 2（★★，约 25 分钟，纯 Python）：手算 vs 矩阵

机器人 base 在 world 中位于 $(3, 1, \text{yaw}{=}\pi)$，相机装在 base 上方 0.5 m、前方 0.1 m。
相机看到目标点在相机系 $(0.5, 0.2, 1.0)$。分别用「手算直觉」和「齐次矩阵」求 world 坐标，
两者必须一致。交付：代码 + 手算过程注释。

### 练习 3（★★，约 30 分钟）：两自由度机械臂 URDF

手写一个 2R 平面机械臂 URDF：`base_link → link1 (revolute) → link2 (revolute) → tool0 (fixed)`，
每个 link 给合理的 visual/inertial，关节限位 $\pm\pi$。在 ROS2 机器上用 `check_urdf` 校验、
joint_state_publisher_gui 驱动。交付：URDF 文件 + `check_urdf` 输出 + RViz 截图描述。

### 练习 4（★★，约 20 分钟）：Xacro 化

把练习 3 的机械臂改成 xacro：杆长做成 `property`，两节臂用同一个 `macro` 生成。
交付：`.urdf.xacro` 文件 + `xacro` 展开后的输出对比（行数变化）。

### 练习 5（★★★，约 40 分钟）：TF 广播+监听节点

写一个 rclpy 节点：以 10 Hz 广播 `base_link → target`（target 沿半径 1 m 的圆运动），
同时监听 `world → base_link`（用 static_transform_publisher 提供），
每收到一次就计算 target 在 world 系下的坐标并打印。交付：节点代码 + 日志片段 +
一段说明：为什么监听要用 `tf2_ros.Buffer` + `TransformListener` 而不是直接订阅 `/tf` 话题。

## 参考答案

<details>
<summary>参考答案</summary>

**练习 1**：

```python
import numpy as np

def check_quat(q, tol=1e-3):
    q = np.asarray(q, dtype=float)
    n = np.linalg.norm(q)
    if n < 1e-9:
        raise ValueError("零四元数不是合法姿态")
    if abs(n - 1.0) > tol:
        print(f"警告: |q|={n:.4f}，已归一化")
    return q / n
```

`(0,0,0,0)` 模长为 0，无法归一化——它不对应任何旋转；TF2 遇到会抛异常或产生 NaN，
污染源常常是「忘了给消息的 quaternion 字段赋值」（默认值全零）。

**练习 2**：手算——base 朝 $-x$ 方向（yaw=π），相机在 world $(3-0.1, 1, 0.5)$；
相机系的目标点 $(0.5, 0.2, 1.0)$ 旋转 π 后为 $(-0.5, -0.2, 1.0)$，相加得 $(2.4, 0.8, 1.5)$。
矩阵：$T_{wc} = T_{wb}T_{bc}$，$p_w = T_{wc}\,p_c$，结果一致。

**练习 3**（骨架）：

```xml
<robot name="arm2r">
  <link name="base_link"/>
  <link name="link1"><visual><geometry><cylinder radius="0.03" length="0.4"/></geometry>
    <origin xyz="0 0 0.2"/></visual> ... </link>
  <joint name="shoulder" type="revolute">
    <parent link="base_link"/><child link="link1"/>
    <origin xyz="0 0 0.1"/><axis xyz="0 1 0"/>
    <limit lower="-3.1416" upper="3.1416" effort="10" velocity="2"/>
  </joint>
  <joint name="elbow" type="revolute">
    <parent link="link1"/><child link="link2"/>
    <origin xyz="0 0 0.4"/><axis xyz="0 1 0"/>
    <limit lower="-3.1416" upper="3.1416" effort="10" velocity="2"/>
  </joint>
  <link name="tool0"/>
  <joint name="tool_fixed" type="fixed">
    <parent link="link2"/><child link="tool0"/><origin xyz="0 0 0.4"/>
  </joint>
</robot>
```

注意 revolute 关节必须带 `<limit>`，否则 `check_urdf` 报错。

**练习 4**：`<xacro:property name="len1" value="0.4"/>`，
`<xacro:macro name="segment" params="name parent len">...</xacro:macro>`，
展开后行数通常从 ~60 行模板变成 ~80 行纯 URDF（宏被实例化）。

**练习 5**（要点）：

```python
self.tf_buffer = tf2_ros.Buffer()
self.tf_listener = tf2_ros.TransformListener(self.tf_buffer, self)
t = self.tf_buffer.lookup_transform("world", "target", rclpy.time.Time())
```

为什么不直接订阅 `/tf`：`/tf` 是原始流，你要自己拼树、管时间缓存、处理乱序；
`Buffer` 帮你维护带时间轴的整棵树并支持插值查询。
</details>

## 延伸阅读

- [官方教程：Introducing TF2](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/Tf2/Introduction-To-Tf2.html) 与 [TF2 教程主页](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/Tf2/Tf2-Main.html)
- [官方教程：URDF 系列](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/URDF/URDF-Main.html)（含从零建模与 xacro 精简两篇）
- [tf2 几何接口文档](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/Tf2/Writing-A-Tf2-Broadcaster-Py.html)（Python 广播器写法）
- 下一讲预告：W24 把这份 URDF 扔进 Gazebo Harmonic，让它有物理、有传感器。